In [ ]:
# Code Block 1: Notebook description
#Notebook description

# This notebook evaluates the performance of a portfolio of assets on a buy-and-hold basis.
# It focuses on how multiple assets interact within the portfolio, rather than assessing a specific mechanical trading strategy.
# buy and hold is a good benchmark for any trading strategy, so it is important to evaluate the performance of a portfolio independently of 
# trading strategies we may want to implement on the individual assets.


In [ ]:
# Code Block 2: Load Libraries
# Load Libraries
import numpy as np
import pandas as pd

import statsmodels
import statsmodels.api as sm
from statsmodels.tsa.stattools import coint
from IPython.display import display
import schwab
from schwab.auth import easy_client
from schwab.client import Client
import re 
import os
import matplotlib.pyplot as plt
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import sys
sys.path.append(r"d:\Coding Projects\Investment Analysis")
import yfinance as yf
#from Quantapp.analytics import TimeSeriesAnalytics as Rolling
from Quantapp.data import MacroDataClient
from Quantapp.secrets import load_project_env, require_secret

load_project_env()

#qc = Rolling()
qe = MacroDataClient()

In [ ]:
# Code Block 3: Define functions & Classes
#Define functions & Classes


#takes a dict of portfolio and their total amounts and directional (short or long value), converts dict to weightings instead of absolute values
def create_weighted_portfolio(portfolio):
    total = sum(abs(amount) for amount in portfolio.values())
    return {ticker: (amount / total) * (1 if direction == 'long' else -1)
            for (ticker, amount), direction in zip(portfolio.items(), ['long' if amount >= 0 else 'short' for amount in portfolio.values()])
    }

def create_weight_dict(portfolio):
    total = sum(abs(amount) for amount in portfolio.values())
    return {ticker: amount / total for ticker, amount in portfolio.items()}

def create_equal_weighted_dict(tickers):
    n = len(tickers)
    if n == 0:
        return {}
    equal_weight = 1 / n
    return {ticker: equal_weight for ticker in tickers}

def normalize_yf_ticker(ticker):
    if not isinstance(ticker, str):
        return ticker
    return ticker.strip().replace('/', '-')

def build_yf_ticker_map(tickers):
    return {ticker: normalize_yf_ticker(ticker) for ticker in tickers}

def zscore_series(series):
    mean = series.mean()
    std = series.std(ddof=0)
    if std == 0 or np.isnan(std):
        return pd.Series(0.0, index=series.index)
    return (series - mean) / std

def z_score(series):
    mean = series.mean()
    std  = series.std(ddof=0)

    if std == 0 or np.isnan(std):
        return pd.Series(0.0, index=series.index)

    z = (series - mean) / std
    return z.replace([np.inf, -np.inf], np.nan)

def sharpe_annualized(series):
    mean = series.mean()
    std  = series.std(ddof=0)
    if std == 0 or np.isnan(std):
        return 0.0
    return (mean / std) * np.sqrt(252)


class Portfolio:
    def __init__(self, ticker_dict ,period='10y', interval='1d', equal_weighted=False, portfolio_name='Portfolio'):
        self.portfolio_name = portfolio_name
        self.period = period
        self.interval = interval
        if equal_weighted:
            self.ticker_dict = create_equal_weighted_dict(list(ticker_dict.keys()))
        else:
            self.ticker_dict = ticker_dict
        self.tickers = list(ticker_dict.keys())
        self.yf_ticker_map = build_yf_ticker_map(self.tickers)
        self.yf_tickers = [self.yf_ticker_map[ticker] for ticker in self.tickers]
        self.data = self.get_data() #close prices of the tickers
        
    def __name__(self):
        return self.portfolio_name
    
    def create_equal_weighted_dict(self):
        return {ticker: 1/len(self.tickers) for ticker in self.tickers}
    
    def rebalance_to_equal_weighted(self):
        self.ticker_dict = self.create_equal_weighted_dict()
        
    def _restore_original_ticker_names(self, close_data):
        if isinstance(close_data, pd.Series):
            close_data = close_data.copy()
            close_data.name = self.tickers[0]
            return close_data
        rename_map = {yf_ticker: ticker for ticker, yf_ticker in self.yf_ticker_map.items()}
        return close_data.rename(columns=rename_map)

    #retrieve raw prices using yf.Ticker as a dataframe
    def retrieve_raw_prices(self, data_type='dataframe'):
        """
        Retrieves historical Close price data for all tickers in ticker_dict using yfinance.

        Returns:
            pd.DataFrame: A DataFrame containing Close prices with dates as index and tickers as columns.
        """
        # Download historical data for all tickers
        data = yf.download(
            tickers=self.yf_tickers,
            period=self.period,
            interval=self.interval,
            auto_adjust=True,
            threads=True,
            progress=False
        )
        
        close_data = self._restore_original_ticker_names(data['Close'])

        if data_type == 'dataframe':
            return close_data
        elif data_type == 'dict':
            if isinstance(close_data, pd.Series):
                return {self.tickers[0]: close_data}
            return {ticker: close_data[ticker] for ticker in self.tickers}
        else:
            raise ValueError('data_type should be either dataframe or dict')

    def get_data(self):
        """
        Retrieves historical Close price data for all tickers in ticker_dict using yfinance.

        Returns:
            pd.DataFrame: A DataFrame containing Close prices with dates as index and tickers as columns.
        """
        # Download historical data for all tickers
        data = yf.download(
            tickers=self.yf_tickers,
            period=self.period,
            interval=self.interval,
            auto_adjust=True,
            threads=True,
            progress=False
        )
        
        return self._restore_original_ticker_names(data['Close'])

    def returns(self, n=1, return_type='portfolio'):
        """
        Calculates the daily returns of the portfolio or individual assets.

        Args:
            n (int): The number of periods over which to calculate returns.
            allocation_type (str): Type of returns to calculate.
                                    - 'portfolio' (default): Returns the portfolio's daily returns.
                                    - 'individual': Returns a DataFrame of each individual asset's daily returns.

        Returns:
            pd.Series or pd.DataFrame: 
                - If allocation_type is 'portfolio', returns a Series of portfolio returns.
                - If allocation_type is 'individual', returns a DataFrame of individual asset returns.
        """
        # Calculate daily percentage change
        daily_returns = self.data.pct_change(n).dropna(how='all')

        if return_type == 'portfolio':
            # Compute portfolio returns by multiplying asset returns with their respective weights
            # and summing them up
            portfolio_returns = daily_returns.dot(list(self.ticker_dict.values()))
            return portfolio_returns
        elif return_type == 'individual':
            # Return individual asset returns as a DataFrame
            return daily_returns
        else:
            raise ValueError("Invalid return_type. Choose 'portfolio' or 'individual'.")
           
    def cumulative_returns(self, n=200, return_type='portfolio'):
        """
        Calculates the cumulative returns of the portfolio or individual assets over the last n days.

        Args:
            n (int): Number of recent days to calculate cumulative returns.
            allocation_type (str): Type of returns to calculate.
                                    - 'portfolio' (default): Returns cumulative returns of the portfolio.
                                    - 'individual': Returns cumulative returns of individual assets.

        Returns:
            pd.Series or pd.DataFrame: 
                - If 'portfolio', returns a Series of cumulative returns.
                - If 'individual', returns a DataFrame of cumulative returns for each asset.
        """

        # Retrieve the returns based on allocation type
        all_returns = self.returns(n=n, return_type=return_type)
        
        # Ensure there are enough data points
        if return_type == 'portfolio':
            if len(all_returns) < n:
                raise ValueError(f"Not enough data to calculate cumulative returns for the last {n} days.")
        elif return_type == 'individual':
            if len(all_returns) < n:
                raise ValueError(f"Not enough data to calculate cumulative returns for the last {n} days.")
        else:
            raise ValueError("Invalid return_type. Choose 'portfolio' or 'individual'.")
        
        # Slice the last n days of returns
        recent_returns = all_returns.iloc[-n:]
        
        if return_type == 'portfolio':
            # Calculate cumulative returns for the portfolio
            cumulative_returns = (1 + recent_returns).cumprod() - 1
            return cumulative_returns
        elif return_type == 'individual':
            # Calculate cumulative returns for each individual asset
            cumulative_returns = (1 + recent_returns).cumprod() - 1
            return cumulative_returns
    
    def rolling_sortino_ratio(self, n=21):
        """
        Calculates the rolling Sortino ratio of the portfolio.

        Args:
            n (int): The number of periods to consider in each rolling window.

        Returns:
            pd.Series: The rolling Sortino ratio of the portfolio.
        """
        # Calculate the daily returns of the portfolio
        returns = self.returns()
        
        # Calculate the daily downside deviation
        downside_deviation = returns.rolling(window=n).apply(lambda x: x[x < 0].std())
        
        # Calculate the rolling Sortino ratio
        rolling_sortino = np.sqrt(n) * returns.rolling(window=n).mean() / downside_deviation
        
        return rolling_sortino

    def rolling_correlation(self, benchmark, n=21):
        """
        Calculates the rolling correlation between the portfolio and a benchmark.

        Args:
            benchmark (pd.Series): The returns of the benchmark.
            n (int): The number of periods to consider in each rolling window.

        Returns:
            pd.Series: The rolling correlation between the portfolio and the benchmark.
        """
        # Calculate the daily returns of the benchmark
        benchmark_returns = benchmark
        
        # Calculate the rolling correlation
        rolling_correlation = self.returns().rolling(window=n).corr(benchmark_returns)
        
        return rolling_correlation
    
    def standard_deviation(self):
        """
        Calculates the standard deviation of the portfolio returns.

        Returns:
            float: The standard deviation of the portfolio returns.
        """
        # Calculate the standard deviation of the portfolio returns
        return self.returns().std()
    
    def benchmark_returns_minus_portfolio_returns(self, benchmark_ticker, n = 1):
        """
        Calculates the difference between the returns of the benchmark and the portfolio.

        Args:
            benchmark (pd.Series): The returns of the benchmark.

        Returns:
            pd.Series: The difference between the returns of the benchmark and the portfolio.
        """
        
        benchmark = yf.Ticker(normalize_yf_ticker(benchmark_ticker)).history(period=self.period, interval=self.interval)['Close'].pct_change(n).dropna()
        #convert benchmark index to portfolio index
        benchmark.index = self.returns(n).index
        # Calculate the difference between the benchmark and the portfolio returns
        excess_returns = benchmark - self.returns(n)
        
        return excess_returns
    
    def plot_rolling_sortino(self, plot_assets=False, n=21):
        """
        Plots the rolling Sortino ratio of the portfolio.
        If plot_assets is True, it will also plot the rolling Sortino ratio of each asset in the portfolio.

        Args:
            plot_assets (bool): Whether to plot the rolling Sortino ratio of each asset in the portfolio.
            n (int): The number of periods to consider in each rolling window.
        """

        import plotly.graph_objects as go
        import numpy as np

        # Calculate the rolling Sortino ratio of the portfolio
        portfolio_sortino = self.rolling_sortino_ratio(n=n)

        # Compute mean and standard deviation of the portfolio's Sortino ratio
        mean_val = portfolio_sortino.mean()
        std_val = portfolio_sortino.std()

        # Create the Plotly figure
        fig = go.Figure()
        
        # Add trace for portfolio Sortino ratio as a dashed line
        fig.add_trace(go.Scatter(
            x=portfolio_sortino.index,
            y=portfolio_sortino,
            mode='lines',
            name=f'{self.portfolio_name} Sortino',
            line=dict(width=4, dash='dash')
        ))

        # Define standard deviation levels and corresponding colors for shading
        std_levels = {
            1: {"color": "yellow", "opacity": 0.5},
            2: {"color": "LightCoral", "opacity": 0.5},
        }

        # Add mean and standard deviation lines using a loop to avoid redundancy
        for i in range(-3, 4):
            if i == 0:
                # Add mean line
                fig.add_hline(
                    y=mean_val,
                    line_dash="dash",
                    line_color="Green",
                    line_width=2,
                    annotation_text="Mean",
                    annotation_position="top left"
                )
            else:
                # Add lines for +/-1ÃƒÂÃ†â€™, +/-2ÃƒÂÃ†â€™, +/-3ÃƒÂÃ†â€™
                fig.add_hline(
                    y=mean_val + i * std_val,
                    line_dash="dash",
                    line_color="Red",
                    line_width=2,
                    annotation_text=f"{'+' if i > 0 else ''}{i}ÃƒÂÃ†â€™",
                    annotation_position="top left"
                )

        # Add shaded regions between standard deviation levels
        for i in std_levels:
            # Shaded region between +iÃƒÂÃ†â€™ and +(i+1)ÃƒÂÃ†â€™
            fig.add_shape(
                type="rect",
                x0=portfolio_sortino.index.min(),
                x1=portfolio_sortino.index.max(),
                y0=mean_val + i * std_val,
                y1=mean_val + (i + 1) * std_val,
                line=dict(color="Red", width=2, dash="dash"),
                fillcolor=std_levels[i]["color"],
                opacity=std_levels[i]["opacity"]
            )
            # Shaded region between - (i+1)ÃƒÂÃ†â€™ and -iÃƒÂÃ†â€™
            fig.add_shape(
                type="rect",
                x0=portfolio_sortino.index.min(),
                x1=portfolio_sortino.index.max(),
                y0=mean_val - (i + 1) * std_val,
                y1=mean_val - i * std_val,
                line=dict(color="Red", width=2, dash="dash"),
                fillcolor=std_levels[i]["color"],
                opacity=std_levels[i]["opacity"]
            )
            
        # Add annotation to the center of the plot  
        fig.add_annotation(
            x=portfolio_sortino.index[len(portfolio_sortino) // 2],
            y=mean_val,
            text=f'Portfolio Mean: {mean_val:.2f}',
            showarrow=False,
            yshift=10
        )

        # Add annotations for each shaded region
        for i in std_levels:
            fig.add_annotation(
            x=portfolio_sortino.index[len(portfolio_sortino) // 2],
            y=mean_val + (i + 0.5) * std_val,
            text=f'Portfolio +{i}ÃƒÂÃ†â€™ to +{i+1}ÃƒÂÃ†â€™',
            showarrow=False,
            yshift=10
            )
            fig.add_annotation(
            x=portfolio_sortino.index[len(portfolio_sortino) // 2],
            y=mean_val - (i + 0.5) * std_val,
            text=f'Portfolio -{i+1}ÃƒÂÃ†â€™ to -{i}ÃƒÂÃ†â€™',
            showarrow=False,
            yshift=10
            )
        
            

        # Optionally add traces for each asset's Sortino ratio as solid lines
        if plot_assets:
            for ticker in self.tickers:
                # Calculate asset returns
                asset_returns = self.data[ticker].pct_change().dropna()

                # Calculate downside deviation for rolling window
                downside_deviation = asset_returns.rolling(window=n).apply(lambda x: x[x < 0].std())

                # Calculate rolling Sortino ratio
                rolling_sortino = np.sqrt(n) * asset_returns.rolling(window=n).mean() / downside_deviation

                # Add asset Sortino ratio trace
                fig.add_trace(go.Scatter(
                    x=rolling_sortino.index,
                    y=rolling_sortino,
                    mode='lines',
                    name=f'{ticker} Sortino',
                    line=dict(width=2)
                ))

        # Add a horizontal line at zero
        fig.add_hline(
            y=0,
            line=dict(color="Red", width=2)
        )
        
        # Create a dropdown menu, the values should be 1 year, 3 years, 5 years, 10 years. Default value should be 3 years
        # Create a dropdown menu, the values should be 1 year, 3 years, 5 years, 10 years. Default value should be 3 years
        fig.update_layout(
            updatemenus=[
            dict(
            buttons=[
            dict(
                args=[{"xaxis.range": [portfolio_sortino.index[-252], portfolio_sortino.index[-1]]}],
                label="1 Year",
                method="relayout"
            ),
            dict(
                args=[{"xaxis.range": [portfolio_sortino.index[-756], portfolio_sortino.index[-1]]}],
                label="3 Years",
                method="relayout"
            ),
            dict(
                args=[{"xaxis.range": [portfolio_sortino.index[-1260], portfolio_sortino.index[-1]]}],
                label="5 Years",
                method="relayout"
            ),
            dict(
                args=[{"xaxis.range": [portfolio_sortino.index[0], portfolio_sortino.index[-1]]}],
                label="10 Years",
                method="relayout"
            )
            ],
            direction="down",
            pad={"r": 10, "t": 10},
            showactive=True,
            x=0.1,
            xanchor="left",
            y=1.25,
            yanchor="top",
            active=1  # Set default to 3 years
            ),
            ]
        )

        # Set the default view to 3 years
        fig.update_xaxes(range=[portfolio_sortino.index[-756], portfolio_sortino.index[-1]])

        
        # Update the layout of the plot for better readability and aesthetics
        fig.update_layout(
            title=f'Rolling Sortino Ratio of {self.portfolio_name}',
            xaxis_title='Date',
            yaxis_title='Rolling Sortino Ratio',
            legend_title='Ticker',
            template='plotly_dark',
            height=1000
        )

        # Display the plot
        fig.show()

In [ ]:
# Code Block 4: Define parameters
#Define parameters
time_frame_week = 7
time_frame_short = 21
time_frame_mid   = 50
time_frame_long = 200

selected_time_frame = time_frame_long

CLIENT_ID = require_secret("SCHWAB_CLIENT_ID")
APP_SECRET = require_secret("SCHWAB_APP_SECRET")
CALLBACK_URL = os.getenv("SCHWAB_CALLBACK_URL", "https://127.0.0.1:8182")
TOKEN_PATH = os.getenv("SCHWAB_TOKEN_PATH", "schwab_token.json")
#callback
period = '20y'
interval = '1d'
benchmark_str = 'SPY'

In [ ]:
# Code Block 5: Login to client (must be done manually, you will be prompted for the u...
#Login to client (must be done manually, you will be prompted for the url)
from schwab.auth import client_from_manual_flow
from schwab.client import Client
import json
client = client_from_manual_flow(
    api_key=CLIENT_ID,
    app_secret=APP_SECRET,
    callback_url=CALLBACK_URL,
    token_path=TOKEN_PATH
)

In [ ]:
# Code Block 6: retreive  account and market data
#retreive  account and market data

#account information: the account infromation of course
#positions_df:        the accounts current options positions
#invested_sybmols:     all of the symbols im currently invested in
#total margin:        the total amount of money under my control for each position
#benchmark_close      the historical closing prices of the benchmark data
#portfolio_close      the historical closing prices of the portfolio
#net_direction        the "net direction" of the options portfolio per position


#general account information
account_information = client.get_accounts().json()

#retrieve options options
#-------------------------------------------------------------------------------
acct_map = client.get_account_numbers().json()
acct_hash = acct_map[0]['hashValue']
resp = client.get_account(acct_hash, fields=Client.Account.Fields.POSITIONS)
acct = resp.json()
positions = acct.get("securitiesAccount", {}).get("positions", [])
#-------------------------------------------------------------------------------

#parses positions for all options positions and formats into a simple dataframe
#-------------------------------------------------------------------------------
for position in positions:
    position_symbol = position['instrument'].get('symbol')
    position_symbol = "".join(position_symbol.split())

    positions_df = pd.DataFrame()
    if 'option_metadata_rows' not in locals():
        option_metadata_rows = []
        option_pattern = re.compile(r'^(?P<underlying>[A-Z]{1,6})(?P<expiration>\d{6})(?P<option_type>[CP])(?P<strike>\d{8})$')

    if position.get('instrument', {}).get('assetType') == 'OPTION':
        match = option_pattern.match(position_symbol)
        if match:
            expiration_code = match.group('expiration')
            expiration_date = pd.to_datetime('20' + expiration_code, format='%Y%m%d', errors='coerce')
            strike_price = int(match.group('strike')) / 1000
            option_metadata_rows.append(
                {
                    'symbol': position_symbol,
                    'underlying': match.group('underlying'),
                    'expiration': expiration_date,
                    'option_type': match.group('option_type'),
                    'strike': strike_price,
                    'underlying_symbol': position.get('instrument', {}).get('underlyingSymbol'),
                    'put_call': position.get('instrument', {}).get('putCall'),
                    'long_quantity': position.get('longQuantity', 0.0),
                    'short_quantity': position.get('shortQuantity', 0.0),
                    'net_quantity': position.get('longQuantity', 0.0) - position.get('shortQuantity', 0.0),
                    'average_price': position.get('averagePrice', 0.0),
                }
            )

    positions_df = pd.DataFrame(option_metadata_rows)



#calculates informally the directionality of my portfolio
#---------------------------------------------------------------------------------------------------------
if not positions_df.empty:
    direction_factor = positions_df['option_type'].map({'C': 1, 'P': -1}).fillna(0)
    positions_df['directional_value'] = positions_df['net_quantity'] * positions_df['average_price'] * direction_factor * 100
    net_direction = positions_df.groupby('underlying')['directional_value'].sum()
    sentiment_map = net_direction.apply(lambda x: 'bullish' if x > 0 else ('bearish' if x < 0 else 'neutral'))
    option_sentiment = sentiment_map.to_dict()
    net_direction = net_direction.to_frame('directional_value').join(sentiment_map.rename('sentiment'))
    #display(net_direction.to_frame('directional_value').join(sentiment_map.rename('sentiment')))
else:
    option_sentiment = {}
    
net_direction['sign'] = net_direction['sentiment'].map({'bullish': 1, 'bearish': -1, 'neutral': 0})
#--------------------------------------------------------------------------------------------------------



#retrieve invested symbols
#---------------------------------------------------------------------------------
organized_positions = {}
for pos in positions:
    symbol = pos.get("instrument", {}).get("underlyingSymbol", "")
    if symbol:
        organized_positions.setdefault(symbol, []).append(pos)
        
invested_symbols = list(organized_positions.keys())
#---------------------------------------------------------------------------------


#for each symbol in organized_positions, get the total invested amount
net_invested_amounts = {}
total_margin = {}

for symbol in invested_symbols:
    positions_list = organized_positions[symbol]
    net_sum = 0
    #print(positions_list[0].keys())
    
    for position in positions_list:
        averagePrice = position.get("averagePrice", 0)
        maintenance_requirement = position.get("maintenanceRequirement", 0) / 100
        putCall = position.get("instrument", {}).get("putCall", "N/A")
        short_quantity = position.get("shortQuantity", 0)
        long_quantity = position.get("longQuantity", 0)
        # If it's a PUT, treat price as negative
        if putCall == "PUT":
            averagePrice = -averagePrice

        # Calculate net invested amount using both long and short quantities
        net_invested = (long_quantity - short_quantity) * averagePrice * 100  # times 100 for options
        maintenance_requirement = maintenance_requirement * 100  # times 100 for options
        total_margin[symbol] = total_margin.get(symbol, 0) + maintenance_requirement

        # Print symbol, putCall, averagePrice, short_quantity, long_quantity, and net_invested
        #print(f"Symbol: {symbol}, Put/Call: {putCall}, Average Price: {averagePrice}, Short Quantity: {short_quantity}, Long Quantity: {long_quantity}, Net Invested: {net_invested}")

        net_sum += net_invested
    net_invested_amounts[symbol] = net_sum
    #display(f"Symbol: {symbol}, Net Invested Amount: {net_sum}")
    #display(f"Symbol: {symbol}, Total Margin Requirement: {total_margin[symbol]}")
#-------------------------------------------------------------------------------------

#retrieve core data for benchmark and portfolio
#--------------------------------------------------------------------------
benchmark_data = yf.Ticker('SPY').history(period=period, interval=interval)
invested_symbol_map = build_yf_ticker_map(invested_symbols)
portfolio_data = yf.download(
            tickers=list(invested_symbol_map.values()),
            period=period,
            interval=interval,
            auto_adjust=True,
            threads=True,
            progress=False
        )

portfolio_closing_prices = portfolio_data['Close']
if isinstance(portfolio_closing_prices, pd.Series):
    portfolio_closing_prices.name = invested_symbols[0]
else:
    portfolio_closing_prices = portfolio_closing_prices.rename(columns={yf_ticker: ticker for ticker, yf_ticker in invested_symbol_map.items()})
benchmark_close          = benchmark_data['Close']

#portfolioc_closing prices index is formated like 'YYYY-MM-DD' but benchmark_close index is formated like 'YYYY-MM-DD HH:MM:SS-00:00' 
#need to convert portfolio_closing_prices index to match benchmark_close index

#display(portfolio_closing_prices.index)
portfolio_closing_prices.index = portfolio_closing_prices.index.tz_localize(None)
benchmark_close.index = benchmark_close.index.tz_localize(None)
#--------------------------------------------------------------------------
net_direction
raw_prices = portfolio_closing_prices.copy()


In [ ]:
# Code Block 7: 7) Asset-level signed and unsigned analytics
# =========================
# 7) Asset-level signed and unsigned analytics
# =========================


# Build a sign vector used for signed calculations (default +1 if missing)
signs = (
    net_direction['sign']
    .reindex(daily_returns.columns)
    .fillna(1.0)
)

# Daily return streams: unsigned raw returns and signed tradable returns

# Daily returns (unsigned + signed)
#=========================
daily_returns = portfolio_closing_prices.pct_change().dropna()
daily_returns_signed = daily_returns.mul(signs, axis=1)
#=========================



# =========================
# Assets / Rolling horizon returns (unsigned + signed)
portfolio_assets_rolling_returns_21  = portfolio_closing_prices.pct_change(21).dropna()
portfolio_assets_rolling_returns_50  = portfolio_closing_prices.pct_change(50).dropna()
portfolio_assets_rolling_returns_200 = portfolio_closing_prices.pct_change(200).dropna()

portfolio_assets_rolling_signed_returns_21  = portfolio_assets_rolling_returns_21.mul(signs, axis=1)
portfolio_assets_rolling_signed_returns_50  = portfolio_assets_rolling_returns_50.mul(signs, axis=1)
portfolio_assets_rolling_signed_returns_200 = portfolio_assets_rolling_returns_200.mul(signs, axis=1)

portfolio_assets_rolling_returns_z_scores_21  = portfolio_assets_rolling_returns_21.apply(z_score)
portfolio_assets_rolling_returns_z_scores_50  = portfolio_assets_rolling_returns_50.apply(z_score)
portfolio_assets_rolling_returns_z_scores_200 = portfolio_assets_rolling_returns_200.apply(z_score)

portfolio_assets_rolling_signed_return_z_scores_21  = portfolio_assets_rolling_signed_returns_21.apply(z_score)
portfolio_assets_rolling_signed_return_z_scores_50  = portfolio_assets_rolling_signed_returns_50.apply(z_score)
portfolio_assets_rolling_signed_return_z_scores_200 = portfolio_assets_rolling_signed_returns_200.apply(z_score)

# =========================



# =========================
# Assets / Rolling Sharpe (unsigned + signed)
portfolio_assets_rolling_sharpe_21  = daily_returns.rolling(21).apply(sharpe_annualized)
portfolio_assets_rolling_sharpe_50  = daily_returns.rolling(50).apply(sharpe_annualized)
portfolio_assets_rolling_sharpe_200 = daily_returns.rolling(200).apply(sharpe_annualized)

portfolio_assets_rolling_signed_sharpe_21  = daily_returns_signed.rolling(21).apply(sharpe_annualized)
portfolio_assets_rolling_signed_sharpe_50  = daily_returns_signed.rolling(50).apply(sharpe_annualized)
portfolio_assets_rolling_signed_sharpe_200 = daily_returns_signed.rolling(200).apply(sharpe_annualized)

portfolio_assets_rolling_sharpe_z_scores_21  = portfolio_assets_rolling_sharpe_21.apply(z_score)
portfolio_assets_rolling_sharpe_z_scores_50  = portfolio_assets_rolling_sharpe_50.apply(z_score)
portfolio_assets_rolling_sharpe_z_scores_200 = portfolio_assets_rolling_sharpe_200.apply(z_score)

portfolio_assets_rolling_signed_sharpe_z_scores_21  = portfolio_assets_rolling_signed_sharpe_21.apply(z_score)
portfolio_assets_rolling_signed_sharpe_z_scores_50  = portfolio_assets_rolling_signed_sharpe_50.apply(z_score)
portfolio_assets_rolling_signed_sharpe_z_scores_200 = portfolio_assets_rolling_signed_sharpe_200.apply(z_score)

#=========================

    
#=========================
# Assets / Rolling correlation to benchmark (unsigned + signed)
#=========================
portfolio_assets_rolling_correlation_21  = daily_returns.rolling(21).apply(lambda x: x.corr(benchmark_daily_returns))
portfolio_assets_rolling_correlation_50  = daily_returns.rolling(50).apply(lambda x: x.corr(benchmark_daily_returns))
portfolio_assets_rolling_correlation_200 = daily_returns.rolling(200).apply(lambda x: x.corr(benchmark_daily_returns))

portfolio_assets_rolling_signed_correlation_21  = daily_returns_signed.rolling(21).apply(lambda x: x.corr(benchmark_daily_returns))
portfolio_assets_rolling_signed_correlation_50  = daily_returns_signed.rolling(50).apply(lambda x: x.corr(benchmark_daily_returns))
portfolio_assets_rolling_signed_correlation_200 = daily_returns_signed.rolling(200).apply(lambda x: x.corr(benchmark_daily_returns))

portfolio_assets_rolling_correlation_z_scores_21  = portfolio_assets_rolling_correlation_21.apply(z_score)
portfolio_assets_rolling_correlation_z_scores_50  = portfolio_assets_rolling_correlation_50.apply(z_score)
portfolio_assets_rolling_correlation_z_scores_200 = portfolio_assets_rolling_correlation_200.apply(z_score)

portfolio_assets_rolling_signed_correlation_z_scores_21  = portfolio_assets_rolling_signed_correlation_21.apply(z_score)
portfolio_assets_rolling_signed_correlation_z_scores_50  = portfolio_assets_rolling_signed_correlation_50.apply(z_score)
portfolio_assets_rolling_signed_correlation_z_scores_200 = portfolio_assets_rolling_signed_correlation_200.apply(z_score)   
#=========================




# Assets / Latest z-scores by metric
# =========================

latest_assets_return_z_scores_21  = portfolio_assets_rolling_signed_return_z_scores_21.iloc[-1].sort_values(ascending=False)
latest_assets_return_z_scores_50  = portfolio_assets_rolling_signed_return_z_scores_50.iloc[-1].sort_values(ascending=False)
latest_assets_return_z_scores_200 = portfolio_assets_rolling_signed_return_z_scores_200.iloc[-1].sort_values(ascending=False)

latest_assets_sharpe_z_scores_21  = portfolio_assets_rolling_sharpe_z_scores_21.iloc[-1].sort_values(ascending=False)
latest_assets_sharpe_z_scores_50  = portfolio_assets_rolling_sharpe_z_scores_50.iloc[-1].sort_values(ascending=False)
latest_assets_sharpe_z_scores_200 = portfolio_assets_rolling_sharpe_z_scores_200.iloc[-1].sort_values(ascending=False)

latest_assets_signed_sharpe_z_scores_21  = portfolio_assets_rolling_signed_sharpe_z_scores_21.iloc[-1].sort_values(ascending=False)
latest_assets_signed_sharpe_z_scores_50  = portfolio_assets_rolling_signed_sharpe_z_scores_50.iloc[-1].sort_values(ascending=False)
latest_assets_signed_sharpe_z_scores_200 = portfolio_assets_rolling_signed_sharpe_z_scores_200.iloc[-1].sort_values(ascending=False)


# Display labels: add parentheses only for signed series with negative direction
def format_ticker(ticker, sign):
    return f'({ticker})' if sign < 0 else ticker

latest_assets_return_z_scores_21.index  = [format_ticker(t, signs.loc[t]) for t in latest_assets_return_z_scores_21.index]
latest_assets_return_z_scores_50.index  = [format_ticker(t, signs.loc[t]) for t in latest_assets_return_z_scores_50.index]
latest_assets_return_z_scores_200.index = [format_ticker(t, signs.loc[t]) for t in latest_assets_return_z_scores_200.index]

latest_assets_sharpe_z_scores_21.index  = [format_ticker(t, signs.loc[t]) for t in latest_assets_sharpe_z_scores_21.index]
latest_assets_sharpe_z_scores_50.index  = [format_ticker(t, signs.loc[t]) for t in latest_assets_sharpe_z_scores_50.index]
latest_assets_sharpe_z_scores_200.index = [format_ticker(t, signs.loc[t]) for t in latest_assets_sharpe_z_scores_200.index]

latest_assets_signed_sharpe_z_scores_21.index  = [format_ticker(t, signs.loc[t]) for t in latest_assets_signed_sharpe_z_scores_21.index]
latest_assets_signed_sharpe_z_scores_50.index  = [format_ticker(t, signs.loc[t]) for t in latest_assets_signed_sharpe_z_scores_50.index]
latest_assets_signed_sharpe_z_scores_200.index = [format_ticker(t, signs.loc[t]) for t in latest_assets_signed_sharpe_z_scores_200.index]

# =========================
# Portfolio aggregates (signed equal-weight daily rebalance)
# =========================
# IMPORTANT: use signed returns if you are actually long/short by net_direction
portfolio_daily_returns = daily_returns_signed.mean(axis=1)

portfolio_equity = (1 + portfolio_daily_returns).cumprod()

portfolio_return_21  = portfolio_equity.pct_change(21).dropna()
portfolio_return_50  = portfolio_equity.pct_change(50).dropna()
portfolio_return_200 = portfolio_equity.pct_change(200).dropna()

portfolio_rolling_sharpe_21  = portfolio_daily_returns.rolling(21).apply(sharpe_annualized).dropna()
portfolio_rolling_sharpe_50  = portfolio_daily_returns.rolling(50).apply(sharpe_annualized).dropna()
portfolio_rolling_sharpe_200 = portfolio_daily_returns.rolling(200).apply(sharpe_annualized).dropna()


# =========================
# Benchmark analytics (unsigned benchmark series)
# =========================
benchmark_daily_returns = benchmark_close.pct_change().dropna()
benchmark_equity = (1 + benchmark_daily_returns).cumprod()

benchmark_rolling_sharpe_21  = benchmark_daily_returns.rolling(21).apply(sharpe_annualized).dropna()
benchmark_rolling_sharpe_50  = benchmark_daily_returns.rolling(50).apply(sharpe_annualized).dropna()
benchmark_rolling_sharpe_200 = benchmark_daily_returns.rolling(200).apply(sharpe_annualized).dropna()

portfolio_rolling_signed_correlation_21 = portfolio_daily_returns.rolling(21).apply(lambda x: x.corr(benchmark_daily_returns)).dropna()
portfolio_rolling_signed_correlation_50 = portfolio_daily_returns.rolling(50).apply(lambda x: x.corr(benchmark_daily_returns)).dropna()
portfolio_rolling_signed_correlation_200 = portfolio_daily_returns.rolling(200).apply(lambda x: x.corr(benchmark_daily_returns)).dropna()


In [ ]:
# Code Block 8: 8) Plots
# =========================
# 8) Plots
# =========================
#plot portfolio vs benchmark equity curve
fig = go.Figure()
fig.add_trace(go.Scatter(x=portfolio_equity.index, y=portfolio_equity, mode='lines', name='Portfolio Equity Curve'))
fig.add_trace(go.Scatter(x=benchmark_equity.index, y=benchmark_equity, mode='lines', name='Benchmark (SPY) Equity Curve'))
fig.update_layout(title='Portfolio vs Benchmark Equity Curve', xaxis_title='Date', yaxis_title='Equity Curve', template='plotly_dark')
fig.show()

#plot portfolio vs benchmark rolling sharpe
fig = go.Figure()
fig.add_trace(go.Scatter(x=portfolio_rolling_sharpe_21.index, y=portfolio_rolling_sharpe_21, mode='lines', name='Portfolio Rolling Sharpe (21 days)'))
fig.add_trace(go.Scatter(x=benchmark_rolling_sharpe_21.index, y=benchmark_rolling_sharpe_21, mode='lines', name='Benchmark (SPY) Rolling Sharpe (21 days)'))
fig.update_layout(title='Portfolio vs Benchmark Rolling Sharpe (21 days)', xaxis_title='Date', yaxis_title='Rolling Sharpe', template='plotly_dark')
fig.show()

#plot portfolio rolling correlation , create a dropdown to select between 21, 50, and 200 days with a zero line
fig = go.Figure()
fig.add_trace(go.Scatter(x=portfolio_rolling_signed_correlation_21.index, y=portfolio_rolling_signed_correlation_21, mode='lines', name='Portfolio Rolling Correlation (21 days)'))
fig.add_trace(go.Scatter(x=portfolio_rolling_signed_correlation_50.index, y=portfolio_rolling_signed_correlation_50, mode='lines', name='Portfolio Rolling Correlation (50 days)', visible=False))
fig.add_trace(go.Scatter(x=portfolio_rolling_signed_correlation_200.index, y=portfolio_rolling_signed_correlation_200, mode='lines', name='Portfolio Rolling Correlation (200 days)', visible=False))
fig.update_layout(
    title='Portfolio Rolling Correlation vs Benchmark (SPY)',
    xaxis_title='Date',
    yaxis_title='Rolling Correlation',
    template='plotly_dark',
    updatemenus=[
        dict(
            buttons=[
                dict(
                    args=[{"visible": [True, False, False]}],
                    label="21 Days",
                    method="update"
                ),
                dict(
                    args=[{"visible": [False, True, False]}],
                    label="50 Days",
                    method="update"
                ),
                dict(
                    args=[{"visible": [False, False, True]}],
                    label="200 Days",
                    method="update"
                ),
            ],
            direction="down",
            pad={"r": 10, "t": 10},
            showactive=True,
            x=0.1,
            xanchor="left",
            y=1.15,
            yanchor="top"
        ),
    ]
)
#add a horizontal line at y=0
fig.add_hline(y=0, line=dict(color="Red", width=2, dash="dash"))
fig.show()


In [ ]:
# Code Block 9: Relative Price Strength Z-Score Plots
#Relative Price Strength Z-Score Plots
def plot_z_score_diff_dropdown(z_scores_map, title_metric='Return'):
    diff_frames = {}
    global_zmin = np.inf
    global_zmax = -np.inf
    for label, z_scores in z_scores_map.items():
        diff = pd.DataFrame(
            z_scores.values[:, None] - z_scores.values,
            index=z_scores.index,
            columns=z_scores.index
        ).astype(float)
        diff_frames[label] = diff
        global_zmin = min(global_zmin, diff.min().min())
        global_zmax = max(global_zmax, diff.max().max())

    fig = go.Figure()
    labels = list(diff_frames.keys())
    for idx, label in enumerate(labels):
        diff = diff_frames[label]
        fig.add_trace(go.Heatmap(
            z=diff.values,
            x=diff.columns,
            y=diff.index,
            zmin=global_zmin,
            zmax=global_zmax,
            colorscale='RdBu',
            colorbar=dict(title='Z-Score Diff'),
            text=diff.round(2),
            hovertemplate='%{y} vs %{x}<br>Diff: %{z:.2f}<extra></extra>',
            visible=idx == 0
        ))

    buttons = []
    for idx, label in enumerate(labels):
        visibility = [i == idx for i in range(len(labels))]
        buttons.append(
            dict(
                label=f'{label}-Day',
                method='update',
                args=[
                    {"visible": visibility},
                    {"title": f'{title_metric} Pairwise Z-Score Differences ({label}-day)'}
                ]
            )
        )

    fig.update_layout(
        title=f'{title_metric} Pairwise Z-Score Differences ({labels[0]}-day)',
        template='plotly_dark',
        updatemenus=[dict(
            buttons=buttons,
            direction="down",
            showactive=True,
            x=0.5,
            xanchor="center",
            y=1.2,
            yanchor="top"
        )],
        height=700
    )
    fig.update_xaxes(title_text='Assets')
    fig.update_yaxes(title_text='Assets')
    fig.show()

from Quantapp.analytics import TimeSeriesAnalytics as Rolling, RiskRelativeAnalytics

rolling = Rolling()
risk_relative_analytics = RiskRelativeAnalytics()

time_frame_map = {
    '21': time_frame_short,
    '50': time_frame_mid,
    '200': time_frame_long,
}

sign_series = net_direction['sign'].reindex(portfolio_closing_prices.columns).fillna(1.0).astype(float)

def format_ticker(ticker, sign):
    return f'({ticker})' if sign < 0 else ticker

def format_snapshot_map(snapshot_map, sign_series):
    formatted = {}
    for label, series in snapshot_map.items():
        formatted_series = series.copy()
        formatted_series.index = [format_ticker(ticker, sign_series.loc[ticker]) for ticker in formatted_series.index]
        formatted[label] = formatted_series
    return formatted

def highlight_signed_bars(index, sign_series, default_color='#636EFA', highlight_color='#F59E0B'):
    colors = []
    for ticker in index:
        raw_ticker = ticker.strip('()') if isinstance(ticker, str) else ticker
        sign_value = sign_series.reindex([raw_ticker]).fillna(1.0).iloc[0]
        colors.append(highlight_color if sign_value < 0 else default_color)
    return colors

benchmark_snapshot = risk_relative_analytics.build_multi_asset_benchmark_snapshot(
    analytics=rolling,
    asset_close=portfolio_closing_prices,
    benchmark_close=benchmark_close,
    time_frame_map=time_frame_map,
    sign_map=sign_series,
)

windows_signed = format_snapshot_map(
    benchmark_snapshot['signed_asset_latest_zscores'],
    sign_series,
)
windows_unsigned = benchmark_snapshot['unsigned_asset_latest_zscores']
windows_benchmark_minus_assets_signed = format_snapshot_map(
    benchmark_snapshot['signed_spread_latest_zscores'],
    sign_series,
)
windows_benchmark_minus_assets_unsigned = benchmark_snapshot['unsigned_spread_latest_zscores']
signed_return_zscore_map = format_snapshot_map(
    benchmark_snapshot['signed_return_latest_zscores'],
    sign_series,
)

fig = make_subplots(
    rows=2,
    cols=2,
    shared_xaxes=False,
    vertical_spacing=0.12,
    horizontal_spacing=0.08,
    subplot_titles=(
        'Asset Sharpe Ratio Z-Scores (Signed)',
        'Asset Sharpe Ratio Z-Scores (Unsigned)',
        f'{benchmark_str} - Asset Sharpe Spread Z-Score (Signed)',
        f'{benchmark_str} - Asset Sharpe Spread Z-Score (Unsigned)',
    )
)

window_labels = list(windows_signed.keys())
for idx, window in enumerate(window_labels):
    z_signed = windows_signed[window]
    z_unsigned = windows_unsigned[window]
    z_diff_signed = windows_benchmark_minus_assets_signed[window]
    z_diff_unsigned = windows_benchmark_minus_assets_unsigned[window]
    unsigned_bar_colors = highlight_signed_bars(z_unsigned.index, sign_series)
    unsigned_spread_bar_colors = highlight_signed_bars(z_diff_unsigned.index, sign_series)

    fig.add_trace(
        go.Bar(x=z_signed.index, y=z_signed.values, name=f'{window}-Day Signed', showlegend=False, visible=idx == 0),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Bar(x=z_unsigned.index, y=z_unsigned.values, marker_color=unsigned_bar_colors, name=f'{window}-Day Unsigned', showlegend=False, visible=idx == 0),
        row=1,
        col=2,
    )
    fig.add_trace(
        go.Bar(x=z_diff_signed.index, y=z_diff_signed.values, name=f'{window}-Day {benchmark_str} - Asset Spread', showlegend=False, visible=idx == 0),
        row=2,
        col=1,
    )
    fig.add_trace(
        go.Bar(x=z_diff_unsigned.index, y=z_diff_unsigned.values, marker_color=unsigned_spread_bar_colors, name=f'{window}-Day {benchmark_str} - Asset Spread', showlegend=False, visible=idx == 0),
        row=2,
        col=2,
    )

for r in [1, 2]:
    for c in [1, 2]:
        fig.add_hline(y=0, line_dash='dash', line_color='red', row=r, col=c)
        fig.add_hline(y=1, line_dash='dot', line_color='green', row=r, col=c)
        fig.add_hline(y=-1, line_dash='dot', line_color='green', row=r, col=c)
        fig.add_hline(y=2, line_dash='dot', line_color='orange', row=r, col=c)
        fig.add_hline(y=-2, line_dash='dot', line_color='orange', row=r, col=c)
        fig.add_hline(y=3, line_dash='dot', line_color='purple', row=r, col=c)
        fig.add_hline(y=-3, line_dash='dot', line_color='purple', row=r, col=c)

        fig.add_hrect(y0=-1, y1=1, fillcolor='lightgreen', opacity=0.3, layer='below', line_width=0, row=r, col=c)
        fig.add_hrect(y0=-2, y1=2, fillcolor='lightyellow', opacity=0.3, layer='below', line_width=0, row=r, col=c)
        fig.add_hrect(y0=-3, y1=3, fillcolor='lightcoral', opacity=0.3, layer='below', line_width=0, row=r, col=c)

buttons = []
for idx, window in enumerate(window_labels):
    visibility = [False] * (4 * len(window_labels))
    base = 4 * idx
    visibility[base] = True
    visibility[base + 1] = True
    visibility[base + 2] = True
    visibility[base + 3] = True

    buttons.append(
        dict(
            label=f'{window}-Day',
            method='update',
            args=[
                {'visible': visibility},
                {'title': f'Sharpe Z-Scores + {benchmark_str} Sharpe Spread Z-Scores - {window}-Day'}
            ]
        )
    )

fig.update_layout(
    title=f'Sharpe Z-Scores + {benchmark_str} Sharpe Spread Z-Scores - 21-Day',
    template='plotly_dark',
    updatemenus=[dict(
        buttons=buttons,
        direction='down',
        showactive=True,
        x=0.5,
        xanchor='center',
        y=1.15,
        yanchor='top'
    )],
    height=950
)

fig.update_xaxes(title_text='Assets', row=1, col=1)
fig.update_xaxes(title_text='Assets', row=1, col=2)
fig.update_xaxes(title_text='Assets', row=2, col=1)
fig.update_xaxes(title_text='Assets', row=2, col=2)
fig.update_yaxes(title_text='Z-Score', row=1, col=1)
fig.update_yaxes(title_text='Z-Score', row=1, col=2)
fig.update_yaxes(title_text='Z-Score', row=2, col=1)
fig.update_yaxes(title_text='Z-Score', row=2, col=2)
fig.show()

plot_z_score_diff_dropdown(signed_return_zscore_map, title_metric='Signed Return')







In [ ]:
# Code Block 10: plot historical returns and rolling sharpe ratios
#plot historical returns and rolling sharpe ratios
from plotly.subplots import make_subplots

# plot the 200 day return of the portfolio vs the benchmarks 200 day return
#--------------------------------------------------------------------------
import plotly.graph_objects as go
weighted_portfolio = create_weight_dict(total_margin)

n_values = [21, 50, 200]
default_n = 200

# Build signed weights using direction sentiment
weight_series = pd.Series(weighted_portfolio).reindex(portfolio_closing_prices.columns).fillna(0.0)
sentiment_signs = sentiment_map.reindex(weight_series.index).map({'bullish': 1.0, 'bearish': -1.0, 'neutral': 0.0}).fillna(1.0)
signed_weights = weight_series * sentiment_signs

fig = make_subplots(
    rows=4,
    cols=1,
    shared_xaxes=True,
    subplot_titles=(
        f'{default_n}-Day Return Difference (Z-Score)',
        f'{default_n}-Day Rolling Sharpe Z-Score (Portfolio)',
        f'{default_n}-Day Signed Asset Return Z-Scores',
        f'{default_n}-Day Signed Weighted and Equally Weighted Returns'
    )
)

# Precompute data for each n
traces_data = {}
for n in n_values:
    portfolio_pct = portfolio_closing_prices.pct_change(n).dropna(how='all')
    signed_weighted_returns = portfolio_pct.mul(signed_weights, axis=1)
    portfolio_n_day_return = signed_weighted_returns.sum(axis=1).dropna()
    benchmark_n_day_return = benchmark_close.pct_change(n).reindex(portfolio_n_day_return.index)

    asset_zscores = signed_weighted_returns.apply(zscore_series)

    # Compute equally weighted returns
    equal_weights = pd.Series(1.0 / len(signed_weights), index=signed_weights.index)
    equal_weighted_returns = portfolio_pct.mul(equal_weights, axis=1).sum(axis=1).dropna()

    traces_data[n] = {
        'portfolio': portfolio_n_day_return,
        'benchmark': benchmark_n_day_return,
        'diff': portfolio_n_day_return - benchmark_n_day_return,
        'asset_zscores': asset_zscores,
        'equal_weighted': equal_weighted_returns
    }
    traces_data[n]['diff_zscore'] = zscore_series(traces_data[n]['diff'])

# Compute rolling Sharpe for each n with signed weights
returns = portfolio_closing_prices.pct_change().fillna(0.0)
weighted_portfolio_returns = returns.mul(signed_weights, axis=1).sum(axis=1)
benchmark_returns = benchmark_close.pct_change().fillna(0.0)

sharpe_data = {}
for n in n_values:
    rolling_mean_port = weighted_portfolio_returns.rolling(window=n).mean()
    rolling_std_port = weighted_portfolio_returns.rolling(window=n).std()
    rolling_sharpe_port = rolling_mean_port / rolling_std_port

    rolling_mean_bench = benchmark_returns.rolling(window=n).mean()
    rolling_std_bench = benchmark_returns.rolling(window=n).std()
    rolling_sharpe_bench = rolling_mean_bench / rolling_std_bench

    sharpe_data[n] = {
        'portfolio_sharpe': rolling_sharpe_port,
        'benchmark_sharpe': rolling_sharpe_bench
    }

# Add traces for each n, initially visible only for default_n
trace_indices = {}
for i, n in enumerate(n_values):
    visible = (n == default_n)
    trace_indices[n] = []

    fig.add_trace(
        go.Scatter(
            x=traces_data[n]['diff_zscore'].index,
            y=traces_data[n]['diff_zscore'],
            mode='lines',
            name=f'Portfolio - Benchmark {n}-Day Return Difference (Z-Score)',
            visible=visible
        ),
        row=1,
        col=1
    )
    trace_indices[n].append(len(fig.data) - 1)

    sharpe_zscore = zscore_series(sharpe_data[n]['portfolio_sharpe'])
    fig.add_trace(
        go.Scatter(
            x=sharpe_zscore.index,
            y=sharpe_zscore,
            mode='lines',
            name=f'Portfolio {n}-Day Rolling Sharpe Z-Score',
            visible=visible
        ),
        row=2,
        col=1
    )
    trace_indices[n].append(len(fig.data) - 1)

    asset_zscores = traces_data[n]['asset_zscores']
    for ticker in asset_zscores.columns:
        fig.add_trace(
            go.Scatter(
                x=asset_zscores.index,
                y=asset_zscores[ticker],
                mode='lines',
                name=f'{ticker} {n}-Day Signed Return Z-Score',
                visible=visible
            ),
            row=3,
            col=1
        )
        trace_indices[n].append(len(fig.data) - 1)

    # Add signed weighted returns
    fig.add_trace(
        go.Scatter(
            x=traces_data[n]['portfolio'].index,
            y=traces_data[n]['portfolio'],
            mode='lines',
            name=f'Signed Weighted {n}-Day Returns',
            visible=visible
        ),
        row=4,
        col=1
    )
    trace_indices[n].append(len(fig.data) - 1)

    # Add equally weighted returns
    fig.add_trace(
        go.Scatter(
            x=traces_data[n]['equal_weighted'].index,
            y=traces_data[n]['equal_weighted'],
            mode='lines',
            name=f'Equally Weighted {n}-Day Returns',
            visible=visible
        ),
        row=4,
        col=1
    )
    trace_indices[n].append(len(fig.data) - 1)

# Add horizontal lines for z-score levels
fig.add_hline(y=0, line=dict(color="#aaaaaa", width=2, dash="dash"), row=1, col=1)
for level in [1, 2, 3]:
    fig.add_hline(y=level, line=dict(color="#bbbbbb", width=2, dash="dot"), row=1, col=1)
    fig.add_hline(y=-level, line=dict(color="#bbbbbb", width=2, dash="dot"), row=1, col=1)

fig.add_hline(y=0, line=dict(color="#aaaaaa", width=2, dash="dash"), row=2, col=1)
for level in [1, 2, 3]:
    fig.add_hline(y=level, line=dict(color="#bbbbbb", width=2, dash="dot"), row=2, col=1)
    fig.add_hline(y=-level, line=dict(color="#bbbbbb", width=2, dash="dot"), row=2, col=1)

fig.add_hline(y=0, line=dict(color="#aaaaaa", width=2, dash="dash"), row=3, col=1)
for level in [1, 2, 3]:
    fig.add_hline(y=level, line=dict(color="#bbbbbb", width=2, dash="dot"), row=3, col=1)
    fig.add_hline(y=-level, line=dict(color="#bbbbbb", width=2, dash="dot"), row=3, col=1)

fig.add_hline(y=0, line=dict(color="#aaaaaa", width=2, dash="dash"), row=4, col=1)

# Create dropdown buttons
buttons = []
for n in n_values:
    visibility = [False] * len(fig.data)
    for idx in trace_indices[n]:
        visibility[idx] = True
    buttons.append(
        dict(
            label=f"{n}-Day",
            method="update",
            args=[{
                "visible": visibility,
                "title.text": f"{n}-Day Return of Portfolio vs Benchmark",
                "annotations[0].text": f"{n}-Day Return Difference (Z-Score)",
                "annotations[1].text": f"{n}-Day Rolling Sharpe Z-Score (Portfolio)",
                "annotations[2].text": f"{n}-Day Signed Asset Return Z-Scores",
                "annotations[3].text": f"{n}-Day Signed Weighted and Equally Weighted Returns"
            }]
        )
    )

fig.update_layout(
    updatemenus=[dict(buttons=buttons, direction="down", showactive=True, x=0.1, xanchor="left", y=1.12, yanchor="top")],
    height=1300,
    template='plotly_dark',
    title_text=f'{default_n}-Day Return of Portfolio vs Benchmark'
)

fig.update_yaxes(title_text="Spread Z-Score", row=1, col=1)
fig.update_yaxes(title_text="Sharpe Z-Score", row=2, col=1)
fig.update_yaxes(title_text="Asset Z-Score", row=3, col=1)
fig.update_yaxes(title_text="Returns", row=4, col=1)
fig.update_xaxes(title_text="Date", row=4, col=1)

fig.show()
#--------------------------------------------------------------------------


In [ ]:
# Code Block 11: this tells you to increase capital allocation to the asset. this does...
#this tells you to increase capital allocation to the asset. this does not tell you whether or not its a good investment
from scipy.optimize import minimize

#columns are the tickers and rows are the dates
dataframe = raw_prices

def optimal_allocation(dataframe, risk_free_rate=0.0):
    """
    Calculates the optimal allocation of assets in a portfolio to maximize the Sharpe ratio
    using mean-variance optimization.

    :param dataframe: A DataFrame of asset prices (columns=tickers, rows=dates).
    :param risk_free_rate: The risk-free rate for Sharpe ratio calculation.
    :return: A DataFrame with one column 'weight' and rows as asset names, rounded to 2 decimals.
    """
    import numpy as np
    import pandas as pd
    from scipy.optimize import minimize

    # 1) Compute daily returns from price data
    returns_df = dataframe.pct_change().dropna()

    # 2) Calculate average (annualized) returns (252: trading days/year)
    mean_returns = returns_df.mean() * 252

    # 3) Calculate the annualized covariance matrix
    cov_matrix = returns_df.cov() * 252

    # 4) Define the number of assets
    num_assets = len(dataframe.columns)

    # 5) Objective function to maximize the Sharpe ratio => minimize negative Sharpe
    def neg_sharpe(weights):
        ret = np.dot(weights, mean_returns)
        vol = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
        return -(ret - risk_free_rate) / vol

    # 6) Constraint: sum(weights) = 1
    constraints = ({'type': 'eq', 'fun': lambda w: np.sum(w) - 1})

    # 7) Bounds: no short selling => each weight in [0,1]
    bounds = tuple((0.0, 1.0) for _ in range(num_assets))

    # 8) Initial guess: equal allocation
    init_guess = [1.0 / num_assets] * num_assets

    # 9) Optimization
    result = minimize(
        neg_sharpe,
        init_guess,
        method='SLSQP',
        bounds=bounds,
        constraints=constraints
    )

    # 10) Return a DataFrame of weights, rounded to two decimals
    weights_df = pd.DataFrame(result.x, index=dataframe.columns, columns=['weight'])
    weights_df = weights_df.round(2)
    return weights_df

    

def rolling_optimal_allocation(dataframe, window=50, risk_free_rate=0.0):
    """
    Applies the 'optimal_allocation' function over a rolling window.
    Returns a DataFrame where each row corresponds to the weights 
    computed for that window, indexed by the last date in the window.
    """
    import pandas as pd

    # Compute daily returns (drop the first NaN row)
    returns_df = dataframe.pct_change().dropna()

    # List to store each window's optimal weights
    rolling_weights_list = []

    # We only compute weights once we have 'window' days of data
    for i in range(window, len(returns_df) + 1):
        # Price slice for the current rolling window
        slice_data = dataframe.iloc[i - window : i]

        # Reuse the existing 'optimal_allocation' function
        weights_df = optimal_allocation(slice_data, risk_free_rate=risk_free_rate)

        # Append just the weight values (in the same column order as 'dataframe') to the list
        rolling_weights_list.append(weights_df['weight'].values)

    # Create a DataFrame of rolling weights, indexed by the final date in each window
    rolling_index = returns_df.index[window - 1 :]
    rolling_weights_df = pd.DataFrame(
        rolling_weights_list,
        index=rolling_index,
        columns=dataframe.columns
    )

    return rolling_weights_df
'''
def plot_rolling_portfolio_allocation(results):
    """
    Plots the rolling optimal portfolio allocation with annotations for the latest weights.

    Args:
        results (pd.DataFrame): DataFrame containing the portfolio allocation weights.

    Returns:
        None
    """
    import plotly.graph_objects as go

    # Create the Plotly figure
    fig = go.Figure()

    # Add a trace for each asset's weight
    for col in results.columns:
        fig.add_trace(go.Scatter(
            x=results.index,
            y=results[col],
            mode='lines',
            name=col
        ))

    # Add annotations for the latest weight of each asset
    for col in results.columns:
        latest_date = results.index[-1]
        latest_weight = results[col].iloc[-1]
        fig.add_annotation(
            x=latest_date,
            y=latest_weight,
            text=f"{latest_weight:.2f}",
            xanchor='left',
            yanchor='middle',
            showarrow=False,
            font=dict(color=fig.data[results.columns.get_loc(col)].line.color),
            bgcolor="rgba(255,255,255,0.5)"
        )

    # Update layout with title, axis labels, and template
    fig.update_layout(
        title='Rolling Optimal Portfolio Allocation',
        xaxis_title='Date',
        yaxis_title='Weight',
        template='plotly_dark',
        height=600
    )

    # Adjust layout to make space for annotations on the right
    fig.update_layout(
        margin=dict(r=150)
    )

    # Display the plot
    fig.show()
'''

def plot_rolling_portfolio_allocation_stacked(results, smooth_window=5):
    """Plot rolling weights as a stacked area chart with optional smoothing."""
    import plotly.graph_objects as go

    if results.empty:
        print("No rolling allocations were produced. Check window size or input data.")
        return

    smoothed = results.rolling(window=smooth_window, min_periods=1).mean().dropna(how='all')

    fig = go.Figure()
    for col in smoothed.columns:
        fig.add_trace(
            go.Scatter(
                x=smoothed.index,
                y=smoothed[col],
                stackgroup='weights',
                mode='lines',
                name=col,
                #hovertemplate='%{x|%Y-%m-%d}<br>%{y:.2f}<extra>{}</extra>'.format(col)
            )
        )

    fig.update_layout(
        title='Smoothed Rolling Portfolio Weights',
        xaxis_title='Date',
        yaxis_title='Weight',
        template='plotly_dark',
        hovermode='x unified',
        height=600
    )
    fig.show()
    
    
def plot_rolling_portfolio_allocation(results):
    """
    Plots the rolling optimal portfolio allocation with annotations for the latest weights.

    Args:
        results (pd.DataFrame): DataFrame containing the portfolio allocation weights.

    Returns:
        None
    """
    import plotly.graph_objects as go

    # Create the Plotly figure
    fig = go.Figure()

    # Add a trace for each asset's weight
    for col in results.columns:
        fig.add_trace(go.Scatter(
            x=results.index,
            y=results[col],
            mode='lines',
            name=col
        ))

    # Add annotations for the latest weight of each asset
    for col in results.columns:
        latest_date = results.index[-1]
        latest_weight = results[col].iloc[-1]
        fig.add_annotation(
            x=latest_date,
            y=latest_weight,
            text=f"{latest_weight:.2f}",
            xanchor='left',
            yanchor='middle',
            showarrow=False,
            font=dict(color=fig.data[results.columns.get_loc(col)].line.color),
            bgcolor="rgba(255,255,255,0.5)"
        )

    # Update layout with title, axis labels, and template
    fig.update_layout(
        title='Rolling Optimal Portfolio Allocation',
        xaxis_title='Date',
        yaxis_title='Weight',
        template='plotly_dark',
        height=600
    )

    # Adjust layout to make space for annotations on the right
    fig.update_layout(
        margin=dict(r=150)
    )

    # Display the plot
    fig.show()
    
    
benchmark = yf.download('SPY', period=period, interval=interval, auto_adjust=True, progress=False)['Close']
raw_prices = portfolio_closing_prices.copy()

for ticker in raw_prices.columns:
    if ticker in net_direction.index:
        direction = net_direction.loc[ticker, 'directional_value']
        if direction < 0:  # bearish: flip the price series
            first_price = raw_prices[ticker].iloc[0]
            # Flip returns around the first price
            flipped = (first_price ** 2) / raw_prices[ticker]
            
            # Safety: ensure numeric, no inf/NaN
            flipped = flipped.astype(float)
            flipped.replace([np.inf, -np.inf], np.nan, inplace=True)
            flipped.fillna(method='ffill', inplace=True)
            flipped.fillna(method='bfill', inplace=True)
            
            raw_prices[ticker] = flipped
            
#DROP VXX
raw_prices = raw_prices.drop(columns=['VXX'], errors='ignore')


In [ ]:
# Code Block 12: rolling allocations
#rolling allocations
rolling_optimal_allocations = rolling_optimal_allocation(raw_prices, window=selected_time_frame)
plot_rolling_portfolio_allocation(rolling_optimal_allocations)
plot_rolling_portfolio_allocation_stacked(rolling_optimal_allocations, smooth_window=5)

In [ ]:
# Code Block 13: '''
'''
def rolling_sharpe(prices, window=50, risk_free_rate=0.0):
    """
    Computes rolling Sharpe ratio for a DataFrame of prices.
    """
    returns = prices.pct_change().dropna()

    if np.isscalar(risk_free_rate):
        excess_returns = returns - risk_free_rate / 252
    else:
        rf_aligned = pd.Series(risk_free_rate, index=returns.index)
        excess_returns = returns.sub(rf_aligned, axis=0)

    rolling_mean = excess_returns.rolling(window).mean()
    rolling_std = excess_returns.rolling(window).std().replace(0, np.nan)
    return (rolling_mean / rolling_std) * np.sqrt(252)

# Compute rolling Sharpe
window = selected_time_frame
rolling_sharpe_df = rolling_sharpe(raw_prices, window=window)

# Convert to long format for Plotly
rolling_sharpe_long = (
    rolling_sharpe_df
    .reset_index()
    .rename(columns={'index': 'Date'})
    .melt(id_vars='Date', var_name='Ticker', value_name='Rolling Sharpe')
    .dropna(subset=['Rolling Sharpe'])
)

fig = px.line(
    rolling_sharpe_long,
    x='Date',
    y='Rolling Sharpe',
    color='Ticker',
    title=f'Rolling {window}-Day Sharpe Ratios of Assets in Options Portfolio',
    labels={'Rolling Sharpe': 'Rolling Sharpe Ratio'},
    template='plotly_dark',
    height=600
)
fig.show()
'''